# Team Turquoise: PD model training and feature importance (logit marginal effects + LightGBM SHAP)

**What this notebook does**
1. Loads the v3 walk-forward feature files (built from Chloe and Desmond's preprocessing, fit on train, transform on
   everything else; see `data/manifest.json`).
2. For rounds 1-3 (test FY2008, FY2009, FY2010) trains two models on the same rows:
   * **Logistic regression** (statsmodels, firm-clustered standard errors). Importance = **average marginal effect (AME)**:
     how many percentage points the PD moves, on average, when a feature rises by one standard deviation.
   * **LightGBM in two versions**, same settings (Chloe's): **monotone** (the team's expected-sign constraints) and
     **unconstrained** (free to learn any shape). Importance = **mean |SHAP|**: how far, on average, a feature pushes
     a firm's log-odds of default away from the average firm.
3. Scores each round's test year (AUC, accuracy ratio, log loss, Brier, PD level) with a **prior shift** for the level.
4. Trains the **final models on FY2007-2011** (the harness candidates) and reports their feature importance.
5. Compares the importance rankings (logit vs monotone vs unconstrained), checks every sign against the expected-sign
   table, shows what the constraints cost or buy, and writes a report.

**Lockbox.** Round 4 tests on FY2011, the one-time lockbox (D-087/D-119). Its file sits in `LOCKBOX_do_not_open/`
and is only read if you set `OPEN_LOCKBOX = True`, once, after the team has frozen the model.

**Runtime.** Choose *Runtime > Change runtime type > A100 GPU* (it also gives the most RAM and CPU cores).
LightGBM and statsmodels run on the CPU cores; the GPU is optional (see `USE_GPU`). The full run has not been timed yet.

In [ ]:
# ---------------- CONFIG: edit these ----------------
DRIVE_DIR = "/content/drive/MyDrive/2026-11-18_turquoise_colab"   # the uploaded folder
SMOKE = False          # True = 10% of firms, quick test that everything runs
OPEN_LOCKBOX = False   # True ONLY once, after the model is frozen (D-119)
USE_GPU = False        # True = try LightGBM's CUDA build; falls back to CPU (with a message) if it is unavailable
SHAP_SAMPLE = 200_000  # rows used for SHAP plots on the final model (the importance table uses this many rows too)
BOOTSTRAP_DRAWS = 200
FINAL_TAU = 0.014640906163076605   # D-073 random-walk hedge on the s5 window, from
                                   # experiments/calibration/2026-11-08_chloe_lgbm_v1/2026-11-08_params_for_production.json
ROUND_TAU_RULE = "last_train_year"  # per-round target level: default rate of the last training year (rule b, no test info)
SEED = 42

In [ ]:
import os, sys, json, time, shutil, warnings, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm==4.6.0", "shap", "statsmodels", "tabulate"], check=True)
    WORK = "/content/work"
    os.makedirs(WORK, exist_ok=True)
    if not os.path.exists(f"{WORK}/data"):
        print("copying data from Drive to local disk (faster reads)...")
        shutil.copytree(f"{DRIVE_DIR}/data", f"{WORK}/data")
    DATA = f"{WORK}/data"
    OUT = f"{DRIVE_DIR}/outputs"
else:
    DATA = os.environ.get("TQ_DATA", f"{DRIVE_DIR}/data")
    OUT = os.environ.get("TQ_OUT", f"{DRIVE_DIR}/outputs")
os.makedirs(f"{OUT}/figures", exist_ok=True)
print("data:", DATA, "\noutputs:", OUT)

In [ ]:
import numpy as np, pandas as pd
import lightgbm as lgb, shap, statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 40)
print("lightgbm", lgb.__version__, "| shap", shap.__version__, "| statsmodels", sm.__version__)
try:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("no GPU visible")
print("CPU cores:", os.cpu_count())

MAN = json.load(open(f"{DATA}/manifest.json"))
FEATURES = MAN["feature_columns"]
GROUPS = MAN["feature_groups"]
print(len(FEATURES), "features | team code:", MAN["team_code"])
pd.DataFrame([{k: r[k] for k in ["round", "role", "train_years", "test_year", "train_rows_out", "valid_es_rows_out",
               "test_rows_out", "train_default_rate", "test_default_rate"]} for r in MAN["rounds"]])

## 0. Leakage check on the files (audit finding A1)
In a clean file the industry target encoding in the early-stopping firms and the test year can only take values
from the frozen training mapping, so `share_te_from_frozen_mapping` must be 1.0 and the mean TE stays near the
**training** default rate (not the test year's own rate).

In [ ]:
pd.DataFrame([{"round": r["round"], "train_rate": r["train_default_rate"],
               "test_rate": r["test_default_rate"],
               "mean_TE_test": r["leak_check_test"]["mean_te"],
               "TE_from_frozen_mapping_valid": r["leak_check_valid_es"]["share_te_from_frozen_mapping"],
               "TE_from_frozen_mapping_test": r["leak_check_test"]["share_te_from_frozen_mapping"],
               "train/valid firm overlap": r["train_valid_firm_overlap"]} for r in MAN["rounds"]])

## 1. Helpers: loading, metrics, prior shift

In [ ]:
logit = lambda p: np.log(p / (1 - p))
expit = lambda z: 1 / (1 + np.exp(-z))
FLOOR, CAP = 0.0003, 0.99   # D-075

def load(name, lockbox=False):
    sub = "LOCKBOX_do_not_open/" if lockbox else ""
    df = pd.read_parquet(f"{DATA}/{sub}{name}.parquet")
    if SMOKE:
        keep = pd.Series(df.id.unique()).sample(frac=0.10, random_state=SEED)
        df = df[df.id.isin(keep)]
    return df.reset_index(drop=True)

def calib_slope(y, p):
    z = logit(np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6))
    res = sm.Logit(np.asarray(y), sm.add_constant(z)).fit(disp=0)
    b = np.asarray(res.params)
    return b[1], b[0]

def metrics(y, p, name=""):
    y, p = np.asarray(y), np.clip(np.asarray(p, dtype=float), FLOOR, CAP)
    auc = roc_auc_score(y, p)
    slope, icpt = calib_slope(y, p)
    return {"model": name, "n": len(y), "defaults": int(y.sum()), "AUC": auc, "AR": 2 * auc - 1,
            "log_loss": log_loss(y, p), "brier": brier_score_loss(y, p), "mean_PD": p.mean(),
            "default_rate": y.mean(), "PD/DR": p.mean() / y.mean(), "calib_slope": slope}

def solve_shift(z, target):
    """delta such that mean(expit(z + delta)) == target (bisection)."""
    lo, hi = -10.0, 10.0
    for _ in range(80):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if expit(z + mid).mean() > target else (mid, hi)
    return (lo + hi) / 2

def prior_shift(p_train_insample, tau):
    """D-072: delta = shift that moves the model's own in-sample mean PD on its training rows to tau."""
    return solve_shift(logit(np.clip(p_train_insample, 1e-9, 1 - 1e-9)), tau)

def apply_shift(p, delta):
    return np.clip(expit(logit(np.clip(p, 1e-9, 1 - 1e-9)) + delta), FLOOR, CAP)

def bootstrap_auc_diff(y, p_a, p_b, draws=BOOTSTRAP_DRAWS, seed=SEED):
    """Paired bootstrap of AUC(a) - AUC(b). One row per firm in a test year, so rows = firms."""
    rng = np.random.default_rng(seed); n = len(y); out = []
    y, p_a, p_b = np.asarray(y), np.asarray(p_a), np.asarray(p_b)
    for _ in range(draws):
        i = rng.integers(0, n, n)
        out.append(roc_auc_score(y[i], p_a[i]) - roc_auc_score(y[i], p_b[i]))
    return np.mean(out), np.percentile(out, 2.5), np.percentile(out, 97.5)

## 2. Model recipes
**Logit preparation (all fitted on the training rows of the round only):** drop constant columns, clip continuous
features at the training 1st/99th percentiles (several ratios have extreme values, e.g. the "zero denominator -> 1"
rule turns ratios into euro amounts, audit B3), standardise every feature with the training mean and SD, and drop
one of any pair with |correlation| > 0.98, and drop rare 0/1 flags (rare side < 500 rows or < 10 defaults: the logit
cannot estimate them). Legal-form and region dummies are re-referenced to their largest
category, so each dummy's effect reads "versus the most common legal form / region". After standardising, a coefficient's AME reads as
**"percentage-point change in PD for a one-SD increase"**, so features are comparable.

**LightGBM:** Chloe's configuration (`lightgbm_model.py`): 15 leaves, learning rate 0.03, min 200 rows per leaf,
0.8 column sampling, L2 = 1, early stopping (150 rounds) on the 20% held-aside firms, monotone constraints from the
team's expected-sign map.

In [ ]:
MONOTONE = {"equity_ratio": -1, "st_debt_share": 1, "lt_debt_to_capital": 1, "cash_ratio": -1,
            "cash_to_financing_exp_ratio": -1, "AR_ratio": 1, "payables_ratio": 1, "wc_ratio": -1,
            "cur_debt_ratio": -1, "quick_ratio": -1, "roa": -1, "op_margin": -1, "net_prof_margin": -1,
            "int_coverage": -1, "financing_expense_ratio": 1, "operating_cash_flow_ratio": -1,
            "ebitda_to_debt": -1, "loss_flag": 1, "ateco_sector_te": 1}   # copied from lightgbm_model.py @ 4e0ba3a
LGB_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=15, min_child_samples=200, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)
MAX_TREES = 600 if SMOKE else 5000

def is_binary(s):
    return set(pd.unique(s.dropna())).issubset({0, 1})

class LogitPrep:
    def fit(self, X, y):
        X = X.astype(float); y = np.asarray(y)
        self.cols = [c for c in X.columns if X[c].nunique() > 1]
        self.dropped_constant = [c for c in X.columns if c not in self.cols]
        # Dummy families: the team's reference level is the alphabetically first category, which can be tiny and
        # makes the dummies nearly collinear. Re-reference each family to its LARGEST category instead.
        self.dropped_reference = []
        for prefix in ("legal_", "city_"):
            fam = [c for c in self.cols if c.startswith(prefix)]
            if fam:
                self.dropped_reference.append(max(fam, key=lambda c: X[c].mean()))
        self.cols = [c for c in self.cols if c not in self.dropped_reference]
        # Rare flags (rare side < 500 rows or < 10 defaults) cause near-separation in a logit: drop them from the
        # logit only (LightGBM keeps them).
        self.dropped_rare = []
        for c in self.cols:
            if is_binary(X[c]):
                ones = X[c].values == 1
                rare = ones if ones.sum() <= (~ones).sum() else ~ones
                if rare.sum() < 500 or y[rare].sum() < 10:
                    self.dropped_rare.append(c)
        self.cols = [c for c in self.cols if c not in self.dropped_rare]
        self.binary = [c for c in self.cols if is_binary(X[c])]
        self.lo = X[self.cols].quantile(0.01); self.hi = X[self.cols].quantile(0.99)
        self.lo[self.binary] = 0; self.hi[self.binary] = 1
        Z = X[self.cols].clip(self.lo, self.hi, axis=1)
        self.mu = Z.mean(); self.sd = Z.std().replace(0, 1)
        Z = (Z - self.mu) / self.sd
        corr = Z.sample(min(len(Z), 200_000), random_state=SEED).corr().abs()
        dropped = []
        for i, a in enumerate(self.cols):
            if a in dropped: continue
            for b in self.cols[i + 1:]:
                if b not in dropped and corr.loc[a, b] > 0.98:
                    dropped.append(b)
        self.dropped_collinear = dropped
        self.use = [c for c in self.cols if c not in dropped]
        return self
    def transform(self, X):
        Z = X[self.cols].astype(float).clip(self.lo, self.hi, axis=1)
        return ((Z - self.mu) / self.sd)[self.use]

def fit_logit(train):
    prep = LogitPrep().fit(train[FEATURES], train.label)
    Z = sm.add_constant(prep.transform(train[FEATURES]))
    model = sm.Logit(train.label.values, Z)
    try:
        res = model.fit(method="newton", maxiter=100, disp=0, cov_type="cluster", cov_kwds={"groups": train.id.values})
    except Exception as e:   # singular Hessian etc.
        print("newton failed:", e, "-> bfgs")
        res = model.fit(method="bfgs", maxiter=2000, disp=0, cov_type="cluster", cov_kwds={"groups": train.id.values})
    return prep, res

def predict_logit(prep, res, df):
    return np.asarray(res.predict(sm.add_constant(prep.transform(df[FEATURES]), has_constant="add")))

def logit_importance(prep, res):
    me = res.get_margeff(at="overall", method="dydx").summary_frame()
    me = me.rename(columns={"dy/dx": "AME", "Std. Err.": "se", "Pr(>|z|)": "p_value",
                            "Conf. Int. Low": "ci_low", "Cont. Int. Hi.": "ci_high"})
    me.index.name = "feature"
    me = me[["AME", "se", "p_value", "ci_low", "ci_high"]] * [100, 100, 1, 100, 100]   # pp of PD per 1 SD
    me["coef_logodds_per_SD"] = res.params.drop("const")
    me["binary"] = [f in prep.binary for f in me.index]
    me["AME_0to1_pp"] = np.where(me.binary, me.AME / prep.sd[me.index].values, np.nan)  # approx effect of switching a flag on
    me["abs_AME"] = me.AME.abs()
    me["rank_logit"] = me.abs_AME.rank(ascending=False).astype(int)
    return me.sort_values("abs_AME", ascending=False)

_GPU_OK = None
def lgb_params():
    p = dict(LGB_PARAMS)
    if USE_GPU and _GPU_OK is not False:
        p["device_type"] = "cuda"
    return p

VARIANTS = {"lgb_mono": "LightGBM monotone", "lgb_free": "LightGBM unconstrained"}

def fit_lgb(train, valid=None, n_trees=None, monotone=True):
    global _GPU_OK
    cons = [MONOTONE.get(c, 0) if monotone else 0 for c in FEATURES]
    def _fit(params):
        m = lgb.LGBMClassifier(n_estimators=n_trees or MAX_TREES, monotone_constraints=cons, **params)
        if valid is not None:
            m.fit(train[FEATURES], train.label, eval_set=[(valid[FEATURES], valid.label)], eval_metric="binary_logloss",
                  callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        else:
            m.fit(train[FEATURES], train.label)
        return m
    params = lgb_params()
    if params.get("device_type") == "cuda":
        try:
            m = _fit(params); _GPU_OK = True; return m
        except Exception as e:
            _GPU_OK = False
            print("LightGBM CUDA not usable here, falling back to CPU:", str(e)[:200])
            params = dict(LGB_PARAMS)
    return _fit(params)

def shap_values(model, X):
    """Exact TreeSHAP from LightGBM's own C++ code (pred_contrib). Units: log-odds of default."""
    contrib = model.booster_.predict(X[FEATURES], pred_contrib=True,
                                     num_iteration=model.best_iteration_ or None)
    return pd.DataFrame(contrib[:, :-1], columns=FEATURES, index=X.index), contrib[:, -1]

def shap_importance(sv, X):
    imp = pd.DataFrame({"mean_abs_SHAP": sv.abs().mean()})
    imp["direction_corr"] = [spearmanr(X[c], sv[c]).correlation if X[c].nunique() > 1 else np.nan for c in FEATURES]
    imp["direction"] = np.select([imp.direction_corr > 0.1, imp.direction_corr < -0.1],
                                 ["higher value -> higher PD", "higher value -> lower PD"], "mixed / flat")
    imp["rank_shap"] = imp.mean_abs_SHAP.rank(ascending=False).astype(int)
    imp.index.name = "feature"
    return imp.sort_values("mean_abs_SHAP", ascending=False)

## 3. Walk-forward rounds 1-3 (test FY2008, FY2009, FY2010)
Each round: train the logit and both LightGBM versions on the round's training firms, early-stop LightGBM on the
held-aside firms, set the level with a prior shift to the last training year's default rate (no test information),
score the test year once.

In [ ]:
round_metrics, logit_imps, paired = [], {}, []
shap_imps = {v: {} for v in VARIANTS}
best_iters = {v: {} for v in VARIANTS}
for r in MAN["rounds"]:
    if "lockbox" in r["role"]:
        continue
    k = r["round"]; t0 = time.time()
    tr, va, te = load(f"round_{k:02d}_train"), load(f"round_{k:02d}_valid_es"), load(f"round_{k:02d}_test")
    last_year = max(r["train_years"])
    tau = tr.loc[tr.fs_year == last_year, "label"].mean()   # ROUND_TAU_RULE
    y = te.label.values

    prep, res = fit_logit(tr)
    preds = {"logit": (predict_logit(prep, res, te), prior_shift(predict_logit(prep, res, tr), tau))}
    logit_imps[k] = logit_importance(prep, res)

    for v in VARIANTS:
        m = fit_lgb(tr, va, monotone=(v == "lgb_mono"))
        best_iters[v][k] = m.best_iteration_
        preds[v] = (m.predict_proba(te[FEATURES])[:, 1], prior_shift(m.predict_proba(tr[FEATURES])[:, 1], tau))
        sv, _ = shap_values(m, te)
        shap_imps[v][k] = shap_importance(sv, te)

    for name, (p, d) in preds.items():
        round_metrics.append({"round": k, "test_year": r["test_year"], "level": "raw", **metrics(y, p, name)})
        round_metrics.append({"round": k, "test_year": r["test_year"], "level": f"prior shift (tau={tau:.4%})",
                              **metrics(y, apply_shift(p, d), name)})
    for a, b in [("lgb_mono", "logit"), ("lgb_free", "logit"), ("lgb_free", "lgb_mono")]:
        diff, lo, hi = bootstrap_auc_diff(y, preds[a][0], preds[b][0])
        paired.append({"round": k, "test_year": r["test_year"], "comparison": f"AUC {a} - {b}",
                       "diff": diff, "ci_low": lo, "ci_high": hi})
    print(f"round {k}: trees mono {best_iters['lgb_mono'][k]} / free {best_iters['lgb_free'][k]}; logit dropped "
          f"constant {prep.dropped_constant}, collinear {prep.dropped_collinear}, rare {prep.dropped_rare}, "
          f"reference {prep.dropped_reference}; {time.time() - t0:.0f}s")

RM = pd.DataFrame(round_metrics)
RM.to_csv(f"{OUT}/round_metrics.csv", index=False)
PAIRED = pd.DataFrame(paired); PAIRED.to_csv(f"{OUT}/paired_auc_bootstrap.csv", index=False)
RM.round(4)

In [ ]:
PAIRED.round(4)

## 4. Importance stability across rounds
A credible driver should rank similarly in every year. Big rank jumps = the driver is unstable or collinear.

In [ ]:
def stability(imps, rank_col):
    t = pd.concat({f"round_{k}": v[rank_col] for k, v in imps.items()}, axis=1)
    rounds = t.columns
    t["mean_rank"] = t[rounds].mean(axis=1); t["max_rank_jump"] = t[rounds].max(axis=1) - t[rounds].min(axis=1)
    return t.sort_values("mean_rank")
STAB = {v: stability(shap_imps[v], "rank_shap") for v in VARIANTS}
STAB["logit"] = stability(logit_imps, "rank_logit")
for name, t in STAB.items():
    t.to_csv(f"{OUT}/stability_ranks_{name}.csv")
    print(name); display(t.head(15))

## 5. Round 4 (training FY2007-10): early stopping only
Round 4 is fitted only to learn each LightGBM version's **tree count** for the final model (early stopping on its
held-aside firms, which are FY2007-10 rows). The FY2011 lockbox is **not** read here.

In [ ]:
tr4, va4 = load("round_04_train"), load("round_04_valid_es")
M4 = {}
for v in VARIANTS:
    M4[v] = fit_lgb(tr4, va4, monotone=(v == "lgb_mono"))
    best_iters[v][4] = M4[v].best_iteration_
N_TREES_FINAL = {v: int(best_iters[v][4]) for v in VARIANTS}
print("best iterations by round:", best_iters, "\nfinal model trees:", N_TREES_FINAL)

## 6. Final models on FY2007-2011 (the harness candidates)
* Both LightGBM versions with their round-4 tree counts, then a **prior shift** to `FINAL_TAU` (D-073 hedge).
* Logit on the same rows, for the marginal-effect table.

In [ ]:
fin = load("final_train_FY2007_2011")
prep_f, res_f = fit_logit(fin)
LOGIT_FINAL = logit_importance(prep_f, res_f)
LOGIT_FINAL.to_csv(f"{OUT}/final_logit_marginal_effects.csv")

FINAL_M, DELTA = {}, {}
final_params = {"tau": FINAL_TAU, "floor": FLOOR, "cap": CAP,
                "pd_formula": "clip(expit(logit(p_raw) + delta), floor, cap)",
                "training_rows": len(fin), "training_default_rate": float(fin.label.mean()),
                "logit_dropped_constant": prep_f.dropped_constant, "logit_dropped_rare_flags": prep_f.dropped_rare,
                "logit_reference_dummies (new reference level)": prep_f.dropped_reference,
                "logit_dropped_collinear": prep_f.dropped_collinear}
for v in VARIANTS:
    FINAL_M[v] = fit_lgb(fin, n_trees=N_TREES_FINAL[v], monotone=(v == "lgb_mono"))
    p_fin = FINAL_M[v].predict_proba(fin[FEATURES])[:, 1]
    DELTA[v] = prior_shift(p_fin, FINAL_TAU)
    FINAL_M[v].booster_.save_model(f"{OUT}/final_{v}.txt")
    final_params[v] = {"n_trees": N_TREES_FINAL[v], "r_w_mean_insample_pd": float(p_fin.mean()), "delta": DELTA[v]}
json.dump(final_params, open(f"{OUT}/final_model_params.json", "w"), indent=2)
final_params

In [ ]:
samp = fin.sample(min(SHAP_SAMPLE, len(fin)), random_state=SEED)
SV, SHAP_FIN = {}, {}
for v in VARIANTS:
    SV[v], _ = shap_values(FINAL_M[v], samp)
    SHAP_FIN[v] = shap_importance(SV[v], samp)
    SHAP_FIN[v].to_csv(f"{OUT}/final_{v}_shap_importance.csv")
SHAP_FINAL, SV_F, m_f, DELTA_FINAL = SHAP_FIN["lgb_mono"], SV["lgb_mono"], FINAL_M["lgb_mono"], DELTA["lgb_mono"]
display(SHAP_FIN["lgb_mono"].head(25)); display(SHAP_FIN["lgb_free"].head(25))

In [ ]:
LOGIT_FINAL.head(25)

## 7. Do the methods agree? Logit AME vs monotone SHAP vs unconstrained SHAP
* **Rank agreement**: Spearman correlation between importance measures, and top-15 overlap.
* **Sign check vs the expected-sign table** (+1 = higher value should raise PD):
  * logit: a significant AME with the wrong sign usually means collinearity (another feature carries the effect) or
    a definition problem (e.g. audit B1: `lt_debt_to_capital` for negative-equity firms);
  * unconstrained LightGBM: if the free model learns the wrong direction for a constrained feature, the data does not
    support that constraint as defined. That is the key evidence for keeping or dropping each constraint.

In [ ]:
CMP = (SHAP_FIN["lgb_mono"][["mean_abs_SHAP", "rank_shap", "direction"]]
       .join(SHAP_FIN["lgb_free"][["mean_abs_SHAP", "rank_shap", "direction"]], rsuffix="_free")
       .join(LOGIT_FINAL[["AME", "ci_low", "ci_high", "p_value", "rank_logit"]], how="left"))
CMP["expected_sign"] = [MONOTONE.get(f, 0) for f in CMP.index]
CMP["logit_sign"] = np.sign(CMP.AME)
CMP["logit_sign_conflict"] = (CMP.expected_sign != 0) & (CMP.logit_sign == -CMP.expected_sign) & (CMP.p_value < 0.05)
_dir = {"higher value -> higher PD": 1, "higher value -> lower PD": -1, "mixed / flat": 0}
CMP["free_sign"] = CMP.direction_free.map(_dir)
CMP["free_sign_conflict"] = (CMP.expected_sign != 0) & (CMP.free_sign == -CMP.expected_sign)
both = CMP.dropna(subset=["rank_logit"])
def _agree(a, b, data):
    rho = spearmanr(data[a], data[b]).correlation
    ra, rb = data[a].rank(ascending=False), data[b].rank(ascending=False)
    return rho, len(set(ra.nsmallest(15).index) & set(rb.nsmallest(15).index))
both = both.assign(abs_AME=both.AME.abs())
AGREE = pd.DataFrame([
    {"pair": "logit |AME| vs monotone SHAP", **dict(zip(["spearman", "top15_overlap"], _agree("abs_AME", "mean_abs_SHAP", both)))},
    {"pair": "logit |AME| vs unconstrained SHAP", **dict(zip(["spearman", "top15_overlap"], _agree("abs_AME", "mean_abs_SHAP_free", both)))},
    {"pair": "monotone vs unconstrained SHAP", **dict(zip(["spearman", "top15_overlap"], _agree("mean_abs_SHAP", "mean_abs_SHAP_free", CMP)))},
]).set_index("pair")
RHO = AGREE.loc["logit |AME| vs monotone SHAP", "spearman"]
CMP.to_csv(f"{OUT}/importance_comparison.csv"); AGREE.to_csv(f"{OUT}/importance_agreement.csv")
display(AGREE.round(3))
CMP.sort_values("rank_shap").head(30).round(4)

In [ ]:
print("Logit signs against the expected-sign table (p < 0.05):")
display(CMP[CMP.logit_sign_conflict][["AME", "p_value", "expected_sign", "direction", "direction_free"]].round(4))
print("Unconstrained LightGBM learns the opposite of a team constraint:")
display(CMP[CMP.free_sign_conflict][["expected_sign", "direction_free", "rank_shap_free", "AME", "p_value"]].round(4))

### What the monotone constraints cost or buy
Side by side on the same test years: if the unconstrained model is not clearly better (bootstrap CI includes 0), the
constraints cost nothing in accuracy and buy explainability (every driver moves the PD in the stated direction).

In [ ]:
MONO_VS_FREE = (RM[RM.model.isin(list(VARIANTS))]
                .pivot_table(index=["round", "test_year", "level"], columns="model",
                             values=["AUC", "log_loss", "PD/DR", "calib_slope"]))
MONO_VS_FREE.to_csv(f"{OUT}/monotone_vs_unconstrained.csv")
display(MONO_VS_FREE.round(4))
PAIRED[PAIRED.comparison == "AUC lgb_free - lgb_mono"].round(4)

### Grouped importance (business view)
Sum of mean |SHAP| over the features in each group from the team's feature schema. (SHAP values add up, so group
totals are meaningful; summed |AME| is shown only as a rough logit counterpart.)

In [ ]:
def group_of(f):
    for g, members in GROUPS.items():
        for m in members:
            if f == m or (m.endswith("*") and f.startswith(m[:-1])):
                return g
    return "other"
G = pd.DataFrame({"group": [group_of(f) for f in CMP.index], "SHAP_mono": CMP.mean_abs_SHAP,
                  "SHAP_free": CMP.mean_abs_SHAP_free, "abs_AME_pp": CMP.AME.abs()}, index=CMP.index)
GROUP_IMP = G.groupby("group")[["SHAP_mono", "SHAP_free", "abs_AME_pp"]].sum().sort_values("SHAP_mono", ascending=False)
GROUP_IMP["SHAP_mono_share"] = GROUP_IMP.SHAP_mono / GROUP_IMP.SHAP_mono.sum()
GROUP_IMP["SHAP_free_share"] = GROUP_IMP.SHAP_free / GROUP_IMP.SHAP_free.sum()
GROUP_IMP.to_csv(f"{OUT}/group_importance.csv")
GROUP_IMP.round(4)

## 8. Figures

In [ ]:
def save(fig, name):
    fig.savefig(f"{OUT}/figures/{name}.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(21, 7))
for a, v, col in [(ax[0], "lgb_mono", "#2a9d8f"), (ax[1], "lgb_free", "#8d99ae")]:
    t = SHAP_FIN[v].head(20)
    a.barh(t.index[::-1], t.mean_abs_SHAP[::-1], color=col); a.set_title(f"{VARIANTS[v]}: mean |SHAP| (log-odds)")
tl = LOGIT_FINAL.head(20)
ax[2].barh(tl.index[::-1], tl.AME[::-1], xerr=(tl.AME - tl.ci_low)[::-1], color=np.where(tl.AME[::-1] > 0, "#e76f51", "#457b9d"))
ax[2].axvline(0, color="k", lw=0.5); ax[2].set_title("Logit: AME, pp of PD per +1 SD (95% CI)")
plt.tight_layout(); save(fig, "01_top20_shap_mono_free_logit")

In [ ]:
for v in VARIANTS:
    plt.figure()
    shap.summary_plot(SV[v].values, samp[FEATURES], max_display=20, show=False)
    plt.title(VARIANTS[v]); save(plt.gcf(), f"02_shap_beeswarm_{v}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for a, (x, yv, lab) in zip(axes, [("abs_AME", "mean_abs_SHAP", "logit vs monotone"),
                                  ("abs_AME", "mean_abs_SHAP_free", "logit vs unconstrained"),
                                  ("mean_abs_SHAP", "mean_abs_SHAP_free", "monotone vs unconstrained")]):
    d = both if x == "abs_AME" else CMP
    rx, ry = d[x].rank(ascending=False), d[yv].rank(ascending=False)
    a.scatter(rx, ry, s=12)
    for f in ry.nsmallest(8).index:
        a.annotate(f, (rx[f], ry[f]), fontsize=7)
    a.plot([1, len(d)], [1, len(d)], "k--", lw=0.5)
    a.set_title(f"{lab}: Spearman {spearmanr(d[x], d[yv]).correlation:.2f}")
    a.set_xlabel(f"rank by {x}"); a.set_ylabel(f"rank by {yv}")
plt.tight_layout(); save(fig, "03_rank_agreement")

In [ ]:
# Dependence plots: top-6 monotone features, monotone (teal) vs unconstrained (grey) SHAP on the same firms
top6 = list(CMP.sort_values("rank_shap").index[:6])
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, f in zip(axes.ravel(), top6):
    x = samp[f]; lo, hi = x.quantile([0.01, 0.99]); keep = x.between(lo, hi)
    ax.scatter(x[keep], SV["lgb_free"].loc[keep, f], s=2, alpha=0.2, color="#8d99ae", label="unconstrained")
    ax.scatter(x[keep], SV["lgb_mono"].loc[keep, f], s=2, alpha=0.2, color="#2a9d8f", label="monotone")
    ax.axhline(0, color="k", lw=0.5); ax.set_title(f"{f} (expected sign {MONOTONE.get(f, 0):+d})"); ax.set_ylabel("SHAP (log-odds)")
axes[0, 0].legend(markerscale=5)
plt.tight_layout(); save(fig, "04_shap_dependence_top6_mono_vs_free")

In [ ]:
# Constrained features where the unconstrained model disagrees with the expected sign (if any)
conf = list(CMP[CMP.free_sign_conflict].index)[:6]
if conf:
    fig, axes = plt.subplots(1, len(conf), figsize=(5 * len(conf), 4), squeeze=False)
    for ax, f in zip(axes.ravel(), conf):
        x = samp[f]; lo, hi = x.quantile([0.01, 0.99]); keep = x.between(lo, hi)
        ax.scatter(x[keep], SV["lgb_free"].loc[keep, f], s=2, alpha=0.2, color="#8d99ae")
        ax.scatter(x[keep], SV["lgb_mono"].loc[keep, f], s=2, alpha=0.2, color="#2a9d8f")
        ax.axhline(0, color="k", lw=0.5); ax.set_title(f"{f}: expected {MONOTONE.get(f, 0):+d}")
    plt.tight_layout(); save(fig, "05_constraint_conflicts")
else:
    print("The unconstrained model agrees with every constraint's direction.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, v in zip(axes, VARIANTS):
    ord_ = STAB[v].head(15).index
    for k in shap_imps[v]:
        ax.plot(range(15), shap_imps[v][k].loc[ord_, "rank_shap"], marker="o", label=f"round {k} (test FY{2007 + k})")
    ax.set_xticks(range(15)); ax.set_xticklabels(ord_, rotation=70, fontsize=7); ax.invert_yaxis()
    ax.set_ylabel("SHAP rank"); ax.set_title(f"{VARIANTS[v]}: rank stability across test years")
axes[0].legend(); save(fig, "06_shap_rank_stability")

## 9. Per-firm explanation for a credit officer (monotone model)
Three firms from the final sample (low, median and high PD). Each bar is how much that feature moved the firm's
log-odds of default away from the average firm. Red pushes the PD up, blue pushes it down.

In [ ]:
p_samp = apply_shift(m_f.predict_proba(samp[FEATURES])[:, 1], DELTA_FINAL)
order = np.argsort(p_samp)
examples = {"low PD": order[len(order) // 50], "median PD": order[len(order) // 2], "high PD": order[-len(order) // 200]}
rows = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (lab, i) in zip(axes, examples.items()):
    s = SV_F.iloc[i]; top = s.reindex(s.abs().sort_values(ascending=False).index[:8])
    ax.barh(top.index[::-1], top.values[::-1], color=np.where(top.values[::-1] > 0, "#e76f51", "#457b9d"))
    ax.set_title(f"{lab}: PD {p_samp[i]:.2%}"); ax.axvline(0, color="k", lw=0.5)
    for f in top.index:
        rows.append({"example": lab, "PD": p_samp[i], "feature": f, "value": samp.iloc[i][f], "SHAP_logodds": s[f]})
plt.tight_layout(); save(fig, "07_firm_explanations")
pd.DataFrame(rows).to_csv(f"{OUT}/firm_explanations.csv", index=False)

## 10. Lockbox (FY2011): run ONCE, only after the team freezes the model (D-119)
Uses the round-4 models (trained FY2007-10) and the same recipe as rounds 1-3. Report the result whatever it says.

In [ ]:
if OPEN_LOCKBOX:
    lb = load("round_04_LOCKBOX_FY2011", lockbox=True)
    tau4 = tr4.loc[tr4.fs_year == 2010, "label"].mean()
    prep4, res4 = fit_logit(tr4)
    rows = [metrics(lb.label.values, apply_shift(predict_logit(prep4, res4, lb),
                    prior_shift(predict_logit(prep4, res4, tr4), tau4)), "logit")]
    for v in VARIANTS:
        d4 = prior_shift(M4[v].predict_proba(tr4[FEATURES])[:, 1], tau4)
        rows.append(metrics(lb.label.values, apply_shift(M4[v].predict_proba(lb[FEATURES])[:, 1], d4), v))
    LB = pd.DataFrame(rows)
    LB.to_csv(f"{OUT}/LOCKBOX_FY2011_result.csv", index=False)
    display(LB)
else:
    LB = None
    print("Lockbox closed. Set OPEN_LOCKBOX = True once, after the model is frozen.")

## 11. Write the report

In [ ]:
def md(df, floatfmt=".4f"):
    try:
        return df.to_markdown(floatfmt=floatfmt)
    except ImportError:
        return "```\n" + df.to_string() + "\n```"

lines = [
    "# Feature importance and walk-forward results (v3 split)", "",
    f"Generated by `2026-11-18_feature_importance_colab.ipynb`. SMOKE={SMOKE}. GPU used by LightGBM: {bool(_GPU_OK)}. "
    f"Data: {MAN['team_code']}; split {MAN['validation']}; label {MAN['label']}.", "",
    "## Walk-forward test results (rounds 1-3)", "", md(RM.set_index(["round", "test_year", "model", "level"])), "",
    "Paired bootstrap of AUC differences:", "", md(PAIRED.set_index(["round", "comparison"])), "",
    f"LightGBM best iterations: {best_iters}. Final model trees: {N_TREES_FINAL}.", "",
    "## Monotone vs unconstrained LightGBM", "", md(MONO_VS_FREE), "",
    "Constrained features where the unconstrained model learns the opposite direction:", "",
    md(CMP[CMP.free_sign_conflict][["expected_sign", "direction_free", "rank_shap_free", "AME", "p_value"]]), "",
    "## Final models (FY2007-2011) parameters", "", "```json", json.dumps(final_params, indent=2), "```", "",
    "## LightGBM monotone: mean |SHAP| (final model, top 25)", "", md(SHAP_FIN["lgb_mono"].head(25)), "",
    "## LightGBM unconstrained: mean |SHAP| (final model, top 25)", "", md(SHAP_FIN["lgb_free"].head(25)), "",
    "## Logit: average marginal effects, pp of PD per +1 SD (final model, top 25)", "",
    md(LOGIT_FINAL.head(25).drop(columns=["abs_AME"])), "",
    "## Agreement between the importance measures", "", md(AGREE, ".3f"), "",
    "Logit signs that contradict the expected-sign table (p < 0.05):", "",
    md(CMP[CMP.logit_sign_conflict][["AME", "p_value", "expected_sign", "direction", "direction_free"]]), "",
    "## Grouped importance", "", md(GROUP_IMP), "",
    "## Rank stability across test years", "",
    *[x for v in STAB for x in (f"**{v}**", "", md(STAB[v].head(15), ".1f"), "")],
    "## Lockbox", "", md(LB) if LB is not None else "Not opened.", "",
    "Figures: `figures/`. All tables above are also saved as CSV in this folder.",
]
open(f"{OUT}/REPORT.md", "w").write("\n".join(lines))
print("report written to", f"{OUT}/REPORT.md")